In [5]:
import time
import json

import cst_python as cst
from cst_python.memory_storage import MemoryStorageCodelet

In [6]:
mind = cst.Mind()

surrounding_actions_mo = mind.create_memory_object("SurroundingActions")
skill_manifest_mo = mind.create_memory_object("SkillManifest")
action_command_mo = mind.create_memory_object("ActionCommand")
action_status_mo = mind.create_memory_object("ActionStatus")

mscodelet = MemoryStorageCodelet(mind, host="127.0.0.1")
mscodelet.time_step = 50
mind.insert_codelet(mscodelet)

mind.start()

In [7]:
surrounding_actions_mo.get_info()

[{'name': 'Beber café',
  'description': '',
  'originPosition': [-13.4774513, 1.027, -106.7298],
  'originObject': 'Cafeteira'},
 {'name': 'Usar',
  'description': '',
  'originPosition': [-17.2419643, 0.319925666, -107.341011],
  'originObject': 'Mictório'},
 {'name': 'Trabalhar',
  'description': '',
  'originPosition': [-8.077, 0.842, -105.922],
  'originObject': 'Laptop'}]

In [8]:
json.loads(skill_manifest_mo.get_info())

{'agent': 'Agent',
 'skills': [{'name': 'work',
   'type': 'animation',
   'duration': 9.14053731573,
   'parameters': [],
   'animations': {'agent': 'Sitting Idle', 'hiaac_hub': 'Typing'}},
  {'name': 'drink_coffee',
   'type': 'animation',
   'duration': 6.016666666666,
   'parameters': [],
   'animations': {'agent': 'Drinking'}},
  {'name': 'use_bathroom',
   'type': 'animation',
   'duration': 4.299999999999,
   'parameters': [],
   'animations': {'agent': 'Standing Idle'}},
  {'name': 'walk_to',
   'type': 'navigation',
   'parameters': [{'name': 'destination', 'type': 'vec3'},
    {'name': 'velocity', 'type': 'float'}]}]}

In [9]:
from dataclasses import dataclass, field, asdict
from typing import Any


@dataclass
class CommandEntry:
    Skill : str
    Parameters:dict[str, Any]=field(default_factory=dict)

@dataclass
class CommandPayload:
    Id:int
    Commands:list[CommandEntry]
    

In [10]:
velocity = 3.5

def create_commands(surrounding_actions:dict) -> dict[str, list[CommandEntry]]:
    result = {}

    for action in surrounding_actions:
        commands = []
        position = action["originPosition"]
        name = action["name"]

        commands.append(CommandEntry("walk_to", 
                                    {"destination":position, "velocity":velocity}))

        if name == "Trabalhar":
            commands.append(CommandEntry("work"))

        elif name == "Beber café":
            commands.append(CommandEntry("drink_coffee"))

        elif name == "Usar":
            commands.append(CommandEntry("use_bathroom"))

        result[name] = commands

    return result

In [11]:
commands = create_commands(surrounding_actions_mo.get_info())

In [12]:
payload = CommandPayload(1, commands["Beber café"])

In [23]:
action_status_mo.get_info()

'{"id":1,"index":1,"skill":"drink_coffee","state":"completed"}'

In [15]:
action_command_mo.set_info(asdict(payload))

-1

In [ ]:
last_id = 1
actions = list(commands.keys())

status = action_status_mo.get_info()
last_payload_size = 2

while True:
    status = json.loads(action_status_mo.get_info())
    while status["state"] != "completed" and status["index"] < last_payload_size-1:
        time.sleep(1)
        status = json.loads(action_status_mo.get_info())

    action = random.choice(actions)
    command = commands[action]

    last_id += 1
    last_payload_size = len(command)
    payload = CommandPayload(last_id, command)

    action_command_mo.set_info(asdict(payload))

    while status["id"] != last_id:
        status = json.loads(action_status_mo.get_info())
        time.sleep(1)



KeyboardInterrupt: 

In [42]:
status

{'id': 1, 'index': 1, 'skill': 'drink_coffee', 'state': 'completed'}

In [ ]:
status["state"] != 'completed' and status["index"] < last_payload_size-1

False

In [38]:
status["state"]

'completed'